Cropping

In [ ]:
#pip install tiffslide
#pip install pillow
import os
import tiffslide
from PIL import Image

input_folder = '/16Tbdrive2/subhadeepd/AgeBow/GTEx_Skin_NotSunExposed/Lowest_100_Tissue_Samples'
output_base_folder = '/16Tbdrive2/subhadeepd/AgeBow/GTEx_Skin_NotSunExposed/Cropped_Lowest_100_HEIP'

# Create the base output folder if it doesn't exist
os.makedirs(output_base_folder, exist_ok=True)

# Loop through all .ndpi files in the input folder
for file_name in os.listdir(input_folder):
    if file_name.endswith(''):
        slide_path = os.path.join(input_folder, file_name)
        
        # Create a new folder to save the cropped images for this slide
        output_folder = os.path.join(output_base_folder, file_name.replace('', ''))
        os.makedirs(output_folder, exist_ok=True)

        # Open the .ndpi file using tiffslide
        with tiffslide.open_slide(slide_path) as slide:
            # Get the highest resolution level
            level = 0
            dims = slide.level_dimensions[level]
            width, height = dims

            # Define the tile size (e.g., 1024x1024 pixels)
            tile_size = 1250

            # Loop through the slide and save tiles as PNG images
            for x in range(0, width, tile_size):
                for y in range(0, height, tile_size):
                    # Calculate the region to crop
                    region = (x, y, min(x + tile_size, width), min(y + tile_size, height))
                    # Read the region
                    tile = slide.read_region((x, y), level, (region[2] - region[0], region[3] - region[1]))
                    # Convert to RGB and save as PNG
                    tile = tile.convert("RGB")
                    output_file_name = f'{file_name.replace("", "")}_tile_{x}_{y}.png'
                    tile.save(os.path.join(output_folder, output_file_name))

print("Cropping completed successfully!")

Filtering Cropped

In [ ]:
import cv2
import numpy as np
import os

def is_type1_image(image_path):
    """
    Determines if an image is of type 1 by analyzing its pixel intensity statistics.
    """
    # Load the image
    image = cv2.imread(image_path, cv2.IMREAD_COLOR)
    
    # Convert the image to grayscale
    gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    # Calculate the mean and standard deviation of the pixel values
    mean, std_dev = cv2.meanStdDev(gray_image)
    
    # Threshold values for type 1 images (tuned based on the sample provided)
    mean_threshold = 220  # Adjust this threshold based on your specific images
    std_dev_threshold = 15  # Adjust this threshold based on your specific images
    
    # Determine if the image is type 1
    if mean > mean_threshold and std_dev < std_dev_threshold:
        return True
    return False

def remove_type1_images(directory_path):
    """
    Recursively removes all type 1 images from the specified directory and its subdirectories.
    """
    # Traverse the directory tree
    for root, dirs, files in os.walk(directory_path):
        for filename in files:
            if filename.endswith('.png'):
                file_path = os.path.join(root, filename)
                
                # Check if the image is type 1
                if is_type1_image(file_path):
                    # Remove the file if it is type 1
                    os.remove(file_path)

# Specify the parent directory containing the images
parent_directory_path = '/16Tbdrive2/subhadeepd/AgeBow/GTEx_Skin_NotSunExposed/Cropped_Highest_100_HEIP'

# Remove all type 1 images from the parent directory and its subdirectories
remove_type1_images(parent_directory_path)


Tensor Shape Filtering

In [ ]:
import os
from PIL import Image
import torch
from torchvision import transforms

# Define the desired shape
desired_shape = torch.Size([3, 1250, 1250])

# Transformation to convert images to PyTorch tensors
transform = transforms.ToTensor()

# Directory containing folders with images
base_dir = '/16Tbdrive2/subhadeepd/AgeBow/GTEx_Skin_NotSunExposed/Cropped_Lowest_100_HEIP'

for root, dirs, files in os.walk(base_dir):
    for file in files:
        if file.endswith(('png')):
            file_path = os.path.join(root, file)
            try:
                # Load image
                image = Image.open(file_path)

                # Convert image to tensor
                tensor_image = transform(image)

                # Check the tensor shape
                if tensor_image.shape != desired_shape:
                    print(f"Removing image with shape {tensor_image.shape}: {file_path}")
                    os.remove(file_path)
                else:
                    print(f"Keeping image with shape {tensor_image.shape}: {file_path}")
            except Exception as e:
                print(f"Error processing image {file_path}: {e}")
